# LLM Calls and Structured Outputs

In [1]:
import os
from dotenv import load_dotenv

## Load API Keys

In [2]:
load_dotenv()

LLM_MODEL = os.getenv('LLM_MODEL')
LLM_BASE_URL = os.getenv('LLM_BASE_URL')
GROQ_API_KEY = os.getenv('GROQ_API_KEY')

In [3]:
LLM_MODEL, LLM_BASE_URL, GROQ_API_KEY[:5]

('openai/gpt-oss-120b', 'https://api.groq.com/openai/v1', 'gsk_x')

## Initialize the LLM Client

In [4]:
import langchain
from langchain.chat_models import init_chat_model

In [5]:
model_provider = LLM_MODEL.split('/')[0]
model_provider

'openai'

In [6]:
llm = init_chat_model(
    model=LLM_MODEL,
    model_provider=model_provider,
    api_key=GROQ_API_KEY,
    base_url=LLM_BASE_URL
)

## Make an LLM call

In [7]:
response = llm.invoke('What is the capital of the US?')
print(response.content)

The capital of the United States is **Washington, D.C.**.


In [8]:
print(response.model_dump_json(indent=2))

{
  "content": "The capital of the United States is **Washington, D.C.**.",
  "additional_kwargs": {
    "refusal": null
  },
  "response_metadata": {
    "token_usage": {
      "completion_tokens": 49,
      "prompt_tokens": 79,
      "total_tokens": 128,
      "completion_tokens_details": {
        "accepted_prediction_tokens": null,
        "audio_tokens": null,
        "reasoning_tokens": 25,
        "rejected_prediction_tokens": null,
        "text_tokens": null
      },
      "prompt_tokens_details": null,
      "queue_time": 0.212503886,
      "prompt_time": 0.003692393,
      "completion_time": 0.101755261,
      "total_time": 0.105447654
    },
    "model_provider": "openai",
    "model_name": "openai/gpt-oss-120b",
    "system_fingerprint": "fp_69334c0be3",
    "id": "chatcmpl-fa4c4cbb-be0c-429e-aab7-aebbad7ed012",
    "service_tier": "on_demand",
    "finish_reason": "stop",
    "logprobs": null
  },
  "type": "ai",
  "name": null,
  "id": "lc_run--01a16774-ef34-7990-8363-fd

## Ask for response in Structured Format

In [9]:
from pydantic import BaseModel, Field

In [10]:
class Capital(BaseModel):
    country: str = Field(description='Name of the country')
    capital: str = Field(description='Name of capital city')

In [11]:
llm_with_structured_output = llm.with_structured_output(Capital)

In [12]:
response = llm_with_structured_output.invoke('What is the capital of the US?')
print(response.model_dump_json(indent=2))

{
  "country": "United States",
  "capital": "Washington, D.C."
}


### A more nuanced example

In [13]:
from typing import Literal

TransactionCategory = Literal[
    # --- INFLOWS (Income & Transfers In) ---
    'INCOME_SALARY_RCVD',            # Regular salary, freelancing, active income
    'INVESTMENT_RETURNS_RCVD',       # Interest, stock dividends, fund redemptions
    'INTERNAL_TRANSFER_IN',         # Moving money between your own accounts/cards
    'REFUNDS_AND_REWARDS_RCVD',      # Cashbacks, merchant refunds, tax returns
    'MISC_INFLOW_RCVD',             # Gifts from friends, unclassified incoming cash

    # --- FIXED OUTFLOWS (Bills & Obligations) ---
    'UTILITIES_AND_RENT_PAID',       # Electricity, water, internet, phone, rent
    'DEBT_AND_EMI_PAID',            # Home loans, car loans, credit card minimums
    'INSURANCE_AND_TAX_PAID',       # Medical/auto insurance, municipal/income taxes

    # --- VARIABLE OUTFLOWS (Lifestyle & Daily Living) ---
    'GROCERIES_AND_ESSENTIALS_PAID', # Supermarkets, local vendor produce, pharmacies
    'LIFESTYLE_AND_DINING_PAID',     # Restaurants, swiggy/zomato, movies, shopping, cafes
    'TRAVEL_AND_TRANSPORT_PAID',     # Fuel, Uber/Ola, train tickets, flights, auto repairs
    'INTERNAL_TRANSFER_OUT',        # Moving money out to your own investment/savings apps
    'MISC_OUTFLOW_PAID'             # True anomalies that don't fit anywhere else
]

class Transaction(BaseModel):
    date: str = Field(description='Date in YYYY-MM-DD format')
    amount: float = Field(description='Exact transaction amount as is from Bank Statement')
    description: str = Field(description='Description of Transaction as is from Bank Statement', min_length=3)
    category: TransactionCategory = Field(description='Broad category assigned to the given transaction')

In [14]:
single_transaction = {
    '10th October 2026, $1276.00, Credit Interest Capitalised',
}

In [15]:
llm_with_structured_output = llm.with_structured_output(Transaction)

prompt = f"""
Categorize the bank transaction:
{single_transaction}
""".strip()

response = llm_with_structured_output.invoke(prompt)
print(response.model_dump_json(indent=2))

{
  "date": "2026-10-10",
  "amount": 1276.0,
  "description": "Credit Interest Capitalised",
  "category": "INVESTMENT_RETURNS_RCVD"
}


In [16]:
class TransactionAnalyzerOutput(BaseModel):
    transactions: list[Transaction] = Field('List of categorized transactions')
    confidence: Literal['High', 'Medium', 'Low'] = Field('Overall confidence in categorizing transactions')
    analyst_summary: str = Field('Short summary of your and findings')

In [17]:
transactions = [
    '30/06/2025	Credit Interest Capitalised 	1,177.00',
    '26/03/2026	1265	6.08555E+11	UPI-DR-608554609312-MOHAMMED HARIS J-BKID-806910110000396-taxi 	260',
    '31/03/2026	1265	1.20835E+11	UPI-DR-120835439135-AKSHAYAKALPA FARMS AND FOODS PRIVATE LIMITED-UTIB-1000129101195-WalletRecharge 	1,000.00'
    '31/03/2026	1265	1.20844E+11	UPI-DR-120843961332-PON STORE-YESB-001425000000051-butter milk '
]

In [18]:
llm_with_structured_output = llm.with_structured_output(TransactionAnalyzerOutput)

In [19]:
prompt = f"""
Categorize the bank transactions:
{transactions}
""".strip()

response = llm_with_structured_output.invoke(prompt)
print(response.model_dump_json(indent=2))

{
  "transactions": [
    {
      "date": "2025-06-30",
      "amount": 1177.0,
      "description": "Credit Interest Capitalised",
      "category": "MISC_INFLOW_RCVD"
    },
    {
      "date": "2026-03-26",
      "amount": 260.0,
      "description": "UPI-DR-608554609312-MOHAMMED HARIS J-BKID-806910110000396-taxi",
      "category": "TRAVEL_AND_TRANSPORT_PAID"
    },
    {
      "date": "2026-03-31",
      "amount": 1000.0,
      "description": "UPI-DR-120835439135-AKSHAYAKALPA FARMS AND FOODS PRIVATE LIMITED-UTIB-1000129101195-WalletRecharge",
      "category": "MISC_OUTFLOW_PAID"
    }
  ],
  "confidence": "High",
  "analyst_summary": "Identified three clear transactions: interest credit (inflow), a taxi payment (travel expense), and a wallet recharge (general outflow). Categorization aligns with provided schema."
}
